In [1]:
"""
RSNA Knee Abnormality Detection — Starter Training Pipeline
Targets: ACL, MCL, Medial Meniscus, Lateral Meniscus, Medial OA, Lateral OA,
         PF OA, Effusion, Synovitis, Baker's, Contusion, Fracture
Metric:  Macro-averaged AUC-ROC
"""

import os
import random
import warnings
from pathlib import Path

import albumentations as A
import numpy as np
import pandas as pd
import pydicom
import timm
import torch
import torch.nn as nn
from albumentations.pytorch import ToTensorV2
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from torch.utils.data import DataLoader, Dataset

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
warnings.filterwarnings("ignore")

# Persistent DICOM cache -- a study's already-loaded, resized volume is saved
# once, then reused across every epoch and fold, instead of re-parsing raw
# DICOM files every time. PREBUILT_CACHE_DIR is checked first (a cache built
# and uploaded in a prior session); LOCAL_CACHE_DIR is built fresh this
# session for anything not already cached.
PREBUILT_CACHE_DIR = Path("/kaggle/input/datasets/chiragggg/dicom-cache/dicom_cache")
LOCAL_CACHE_DIR = Path("/kaggle/working/dicom_cache")
LOCAL_CACHE_DIR.mkdir(parents=True, exist_ok=True)


# ── Config ─────────────────────────────────────────────────────────────────────

class CFG:
    # Paths — update DATA_DIR for local use; /kaggle/input/... on Kaggle
    DATA_DIR = Path("/kaggle/input/competitions/rsna-knee-abnormality-detection")
    OUTPUT_DIR = Path("/kaggle/working")

    TARGETS = [
        "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
        "Medial OA", "Lateral OA", "PF OA", "Effusion",
        "Synovitis", "Baker's", "Contusion", "Fracture",
    ]
    NUM_CLASSES = len(TARGETS)  # 12

    # Slices uniformly sampled per series
    N_SLICES = 12

    # Model — currently EfficientNet-B3 (RSNAModel), the proven baseline.
    # RSNAModelV2 (DINOv2 + attention pooling) is kept below, unused, as a
    # real alternative to revisit later -- switch MODEL_NAME and the
    # instantiation line in main() together if you do.
    MODEL_NAME = "efficientnet_b3"
    IMG_SIZE = 224

    # Training
    EPOCHS = 1
    BATCH_SIZE = 4
    GRAD_ACCUM_STEPS = 4  # effective batch size = BATCH_SIZE * GRAD_ACCUM_STEPS
    ENCODER_CHUNK_SIZE = 16  # max slices sent through the encoder at once (caps activation memory)
    LR = 1e-4
    WEIGHT_DECAY = 1e-2
    FOLDS = 2
    SEED = 42

    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    NUM_WORKERS = 4


# ── DICOM Utilities ────────────────────────────────────────────────────────────

def load_dicom_volume(series_dir: Path) -> np.ndarray:
    """Load all slices in a series; sort by InstanceNumber when available."""
    dcm_files = list(series_dir.glob("*.dcm"))

    def sort_key(f):
        try:
            return int(pydicom.dcmread(str(f), stop_before_pixels=True).InstanceNumber)
        except Exception:
            return f.name

    dcm_files.sort(key=sort_key)

    slices = []
    for f in dcm_files:
        dcm = pydicom.dcmread(str(f))
        arr = dcm.pixel_array.astype(np.float32)
        slope = float(getattr(dcm, "RescaleSlope", 1))
        intercept = float(getattr(dcm, "RescaleIntercept", 0))
        slices.append(arr * slope + intercept)

    return np.stack(slices, axis=0)  # (D, H, W)


def normalize_volume(volume: np.ndarray) -> np.ndarray:
    """Clip to 1st/99th percentile then scale to [0, 1]."""
    p1, p99 = np.percentile(volume, [1, 99])
    volume = np.clip(volume, p1, p99)
    volume = (volume - p1) / (p99 - p1 + 1e-6)
    return volume.astype(np.float32)


def get_series_laterality_info(series_dir: Path):
    """Reads just ONE representative DICOM file's header (no pixel data) to
    extract the fields determine_laterality() needs. These fields are
    consistent across a whole series, so one file is enough."""
    dcm_files = list(series_dir.glob("*.dcm"))
    if not dcm_files:
        return None
    dcm = pydicom.dcmread(str(dcm_files[0]), stop_before_pixels=True)
    try:
        return {
            "dicom_tag_laterality": getattr(dcm, "Laterality", None),
            "image_position": [float(x) for x in dcm.ImagePositionPatient],
            "row_cosine": [float(x) for x in dcm.ImageOrientationPatient[:3]],
            "col_cosine": [float(x) for x in dcm.ImageOrientationPatient[3:]],
            "pixel_spacing_col": float(dcm.PixelSpacing[1]),
            "pixel_spacing_row": float(dcm.PixelSpacing[0]),
            "n_cols": int(dcm.Columns),
            "n_rows": int(dcm.Rows),
        }
    except Exception:
        return None  # missing/malformed geometry fields -- treated as ambiguous downstream


def determine_laterality(dicom_tag_laterality, image_position, row_cosine, col_cosine,
                          pixel_spacing_col, pixel_spacing_row, n_cols, n_rows,
                          midline_threshold_mm=15.0):
    """Returns "left", "right", or "ambiguous". Trusts the real DICOM tag
    when present; falls back to real image geometry when it's missing
    (confirmed common -- absent on roughly half of studies in practice)."""
    if dicom_tag_laterality in ("L", "R"):
        return "left" if dicom_tag_laterality == "L" else "right"

    p = np.array(image_position)
    r = np.array(row_cosine)
    d = np.array(col_cosine)
    center = p + r * pixel_spacing_col * (n_cols / 2) + d * pixel_spacing_row * (n_rows / 2)
    cx = center[0]

    if abs(cx) < midline_threshold_mm:
        return "ambiguous"
    return "left" if cx > 0 else "right"


def normalize_laterality(volume: np.ndarray, plane: str, laterality: str,
                          target_convention: str = "left"):
    """Returns (corrected_volume, was_normalized). Coronal/axial: horizontal
    flip, since medial-lateral lies within the image plane. Sagittal: reverse
    slice order instead, since each slice is unchanged by flipping pixels."""
    if laterality == "ambiguous":
        return volume, False
    if laterality == target_convention:
        return volume, True

    if plane in ("coronal", "axial"):
        corrected = volume[:, :, ::-1]
    elif plane == "sagittal":
        corrected = volume[::-1, :, :]
    else:
        raise ValueError(f"Unknown plane: {plane!r}")
    return corrected, True


def sample_slices(volume: np.ndarray, n: int) -> np.ndarray:
    """Uniformly sample n slices along the depth axis."""
    indices = np.linspace(0, volume.shape[0] - 1, n, dtype=int)
    return volume[indices]  # (n, H, W)


def pick_best_series(study_uid: str, series_df: pd.DataFrame) -> str:
    """Prefer sagittal fluid-sensitive series; fall back to first available."""
    rows = series_df[series_df["StudyInstanceUID"] == study_uid]
    preferred = rows[
        (rows["Anatomical_Plane"] == "Sagittal") & (rows["Fluid_Sensitive"] == 1)
    ]
    chosen = preferred if len(preferred) > 0 else rows
    return chosen.iloc[0]["SeriesInstanceUID"]

def get_prioritized_series(study_uid: str, series_df: pd.DataFrame, max_series: int = 3) -> list[str]:
    """
    Returns up to max_series series UIDs for this study, ordered by priority --
    sagittal fluid-sensitive first (reusing your original pick_best_series
    logic), then everything else. Real studies vary in how many series they
    actually have; this handles that gracefully rather than assuming a fixed count.
    """
    rows = series_df[series_df["StudyInstanceUID"] == study_uid]
    preferred = rows[(rows["Anatomical_Plane"] == "Sagittal") & (rows["Fluid_Sensitive"] == 1)]
    others = rows[~rows.index.isin(preferred.index)]
    ordered = pd.concat([preferred, others])
    return ordered["SeriesInstanceUID"].tolist()[:max_series]


# ── Dataset ────────────────────────────────────────────────────────────────────

class RSNADataset(Dataset):
    def __init__(
        self,
        df: pd.DataFrame,
        series_df: pd.DataFrame,
        split: str = "train",    # "train" | "test"
        transform=None,
    ):
        self.df = df.reset_index(drop=True)
        self.series_df = series_df
        self.split = split
        self.transform = transform
        self.series_root = CFG.DATA_DIR / f"{split}_series"

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        import cv2
        row = self.df.iloc[idx]
        study_uid = row["StudyInstanceUID"]

        prebuilt_path = PREBUILT_CACHE_DIR / f"{study_uid}.npy"
        local_path = LOCAL_CACHE_DIR / f"{study_uid}.npy"

        if prebuilt_path.exists():
            all_slices = np.load(prebuilt_path).astype(np.float32)
        elif local_path.exists():
            all_slices = np.load(local_path).astype(np.float32)
        else:
            series_uids = get_prioritized_series(study_uid, self.series_df)
            n_series = len(series_uids)
            slices_per_series = CFG.N_SLICES // n_series
            remainder = CFG.N_SLICES % n_series

            all_slices_list = []
            for i, series_uid in enumerate(series_uids):
                series_dir = self.series_root / study_uid / series_uid
                volume = load_dicom_volume(series_dir)
                volume = normalize_volume(volume)

                # Laterality normalization -- correct medial/lateral image
                # inconsistency before this volume gets used for anything else
                plane_row = self.series_df[
                    (self.series_df["StudyInstanceUID"] == study_uid)
                    & (self.series_df["SeriesInstanceUID"] == series_uid)
                ]
                plane = plane_row.iloc[0]["Anatomical_Plane"].lower() if len(plane_row) > 0 else None
                lat_info = get_series_laterality_info(series_dir)
                if lat_info is not None and plane in ("coronal", "axial", "sagittal"):
                    laterality = determine_laterality(**lat_info)
                    volume, _ = normalize_laterality(volume, plane=plane, laterality=laterality)

                n_this_series = slices_per_series + (1 if i < remainder else 0)
                slices = sample_slices(volume, n_this_series)
                resized_slices = np.stack([
                    cv2.resize(s, (CFG.IMG_SIZE, CFG.IMG_SIZE)) for s in slices
                ])
                all_slices_list.append(resized_slices)

            all_slices = np.concatenate(all_slices_list, axis=0)
            np.save(local_path, all_slices.astype(np.float16))

        images = []
        for s in all_slices:
            img = np.stack([s, s, s], axis=-1)
            if self.transform:
                img = self.transform(image=img)["image"]
            images.append(img)
        images = torch.stack(images, dim=0)
        if self.split == "train":
            raw_labels = row[CFG.TARGETS].values.astype(np.float32)
            valid_mask = (~pd.isnull(raw_labels)).astype(np.float32)
            labels = np.nan_to_num(raw_labels, nan=0.0)

            labels = torch.tensor(labels, dtype=torch.float32)
            valid_mask = torch.tensor(valid_mask, dtype=torch.float32)
            weight = torch.tensor(row[[f"{t}__conf" for t in CFG.TARGETS]].values.astype(np.float32),dtype=torch.float32,)
            return images, labels, valid_mask, weight

        return images, study_uid


# ── Transforms ─────────────────────────────────────────────────────────────────

def get_transforms(is_train: bool) -> A.Compose:
    if is_train:
        return A.Compose([
            A.Resize(CFG.IMG_SIZE, CFG.IMG_SIZE),
            A.HorizontalFlip(p=0.5),
            A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=15, p=0.5),
            A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.3),
            A.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
            ToTensorV2(),
        ])
    return A.Compose([
        A.Resize(CFG.IMG_SIZE, CFG.IMG_SIZE),
        A.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
        ToTensorV2(),
    ])


# ── Model (OLD) — kept, unused, as a fast revert path ───────────────────────────

class RSNAModel(nn.Module):
    """
    2.5D model: encode each slice independently with a CNN backbone,
    then mean-pool slice features before the classification head.
    This is the proven baseline currently instantiated in main().
    """

    def __init__(self):
        super().__init__()
        self.encoder = timm.create_model(
            CFG.MODEL_NAME, pretrained=True, num_classes=0, global_pool="avg"
        )
        if hasattr(self.encoder, "set_grad_checkpointing"):
            self.encoder.set_grad_checkpointing(True)
        feat_dim = self.encoder.num_features
        self.head = nn.Sequential(
            nn.Linear(feat_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, CFG.NUM_CLASSES),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, N, C, H, W = x.shape
        flat = x.view(B * N, C, H, W)
        feats = torch.cat(
            [self.encoder(chunk) for chunk in flat.split(CFG.ENCODER_CHUNK_SIZE, dim=0)],
            dim=0,
        )
        feats = feats.view(B, N, -1).mean(dim=1)       # mean over slices → (B, feat_dim)
        return self.head(feats)                         # (B, num_classes)


# ── Model (NEW) — MERGED from rsna_model_v2_attention.py ───────────────────────

class AttentionPool(nn.Module):
    """
    Takes N slice-level feature vectors, learns which ones matter most for
    THIS study, and returns one weighted-combination feature vector.
    Verified correct (numpy cross-check): softmax over the slice dimension,
    weights sum to 1 per study, genuine weighted combination confirmed.
    """

    def __init__(self, feat_dim: int, hidden_dim: int = 128):
        super().__init__()
        self.attention = nn.Sequential(
            nn.Linear(feat_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, slice_features: torch.Tensor) -> torch.Tensor:
        """slice_features: (B, N, feat_dim). Returns: (B, feat_dim)."""
        raw_scores = self.attention(slice_features)  # (B, N, 1)
        scores = raw_scores.squeeze(-1)               # (B, N)
        weights = torch.softmax(scores, dim=1)         # softmax across slices, per study
        pooled = torch.bmm(weights.unsqueeze(1), slice_features).squeeze(1)  # (B, feat_dim)
        return pooled


class RSNAModelV2(nn.Module):
    """
    RSNA Knee model with DINOv2 backbone and attention-based slice pooling.
    Not currently used in main() -- RSNAModel (EfficientNet-B3) is the proven
    baseline in active use. Kept here as a real, tested alternative to
    revisit later; switch MODEL_NAME and the instantiation line in main()
    together if you do.
    """

    def __init__(self):
        super().__init__()
        self.encoder = timm.create_model(
            CFG.MODEL_NAME, pretrained=True, num_classes=0, img_size=CFG.IMG_SIZE
        )
        feat_dim = self.encoder.num_features
        self.pool = AttentionPool(feat_dim)
        self.head = nn.Sequential(
            nn.Linear(feat_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, CFG.NUM_CLASSES),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, N, C, H, W = x.shape
        flat = x.view(B * N, C, H, W)
        feats = torch.cat(
            [self.encoder(chunk) for chunk in flat.split(CFG.ENCODER_CHUNK_SIZE, dim=0)],
            dim=0,
        )
        feats = feats.view(B, N, -1)          # (B, N, feat_dim) — NOT pooled yet
        pooled = self.pool(feats)               # attention pooling replaces .mean(dim=1)
        return self.head(pooled)                # (B, num_classes)


# ── Metrics ────────────────────────────────────────────────────────────────────

def macro_auc(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """Macro AUC-ROC; skips any target with only one unique class in the batch."""
    aucs = []
    for i in range(y_true.shape[1]):
        if len(np.unique(y_true[:, i])) > 1:
            aucs.append(roc_auc_score(y_true[:, i], y_pred[:, i]))
    return float(np.mean(aucs)) if aucs else 0.0


# ── Train / Validate ───────────────────────────────────────────────────────────

def train_one_epoch(model, loader, optimizer, criterion, scaler, device):
    model.train()
    total_loss = 0.0
    optimizer.zero_grad(set_to_none=True)
    for step, (images, labels, valid_mask, weight) in enumerate(loader):
        images, labels, valid_mask, weight = images.to(device), labels.to(device), valid_mask.to(device), weight.to(device)
        logits = model(images)
        if step == 0:
            with torch.no_grad():
                probs = torch.sigmoid(logits)
                print(f"Prediction std across batch: {probs.std(dim=0)}")
                print(f"Min: {probs.min(dim=0).values}, Max: {probs.max(dim=0).values}")
        raw_loss = criterion(logits, labels)              # shape (batch, 12)
        raw_loss = raw_loss * valid_mask                   # zero out anything that was null
        raw_loss = raw_loss * weight
        #raw_loss = raw_loss * weight.unsqueeze(1)          # scale each sample by its source confidence

        loss = raw_loss.sum() / valid_mask.sum().clamp(min=1) / CFG.GRAD_ACCUM_STEPS
        loss.backward()

        if step == 0:
            encoder_grad_norm = 0.0
            head_grad_norm = 0.0
            for name, param in model.named_parameters():
                if param.grad is not None:
                    if 'encoder' in name:
                        encoder_grad_norm += param.grad.norm().item()
                    else:
                        head_grad_norm += param.grad.norm().item()
            print(f"Encoder gradient norm: {encoder_grad_norm:.4f}")
            print(f"Head+pool gradient norm: {head_grad_norm:.4f}")
        if (step + 1) % CFG.GRAD_ACCUM_STEPS == 0:
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)
        total_loss += loss.item() * CFG.GRAD_ACCUM_STEPS
        if step == 0:
            print("valid_mask sum:", valid_mask.sum().item(), "of", valid_mask.numel())
            print("raw_loss sum before mask:", raw_loss.sum().item())
            #print("raw_loss sum after mask+weight:", (raw_loss * valid_mask * weight.unsqueeze(1)).sum().item())
            print("raw_loss sum after mask+weight:", (raw_loss * valid_mask * weight).sum().item())
    return total_loss / len(loader)


@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    all_preds, all_labels = [], []
    for images, labels, valid_mask, weight in loader:
        images, labels, valid_mask, weight = images.to(device), labels.to(device), valid_mask.to(device), weight.to(device)
        with torch.amp.autocast("cuda"):
            logits = model(images)
            raw_loss = criterion(logits, labels)
            raw_loss = raw_loss * valid_mask
            batch_loss = raw_loss.sum() / valid_mask.sum().clamp(min=1)
            total_loss += batch_loss.item()
        all_preds.append(torch.sigmoid(logits).cpu().numpy())
        all_labels.append(labels.cpu().numpy())
    preds = np.concatenate(all_preds)
    labels = np.concatenate(all_labels)
    return total_loss / len(loader), macro_auc(labels, preds)


# ── Main ───────────────────────────────────────────────────────────────────────

def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def main():
    seed_everything(CFG.SEED)
    CFG.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    train_df = pd.read_csv(CFG.DATA_DIR / "train.csv")
    series_df = pd.read_csv(CFG.DATA_DIR / "train_series.csv")
    hard_labeled_df = train_df.dropna(subset=CFG.TARGETS).copy()
    for col in CFG.TARGETS:
        hard_labeled_df[col] = hard_labeled_df[col].astype(float)
    for target in CFG.TARGETS:
        hard_labeled_df[f"{target}__conf"] = 1.0

    import json
    #with open("/kaggle/input/datasets/chiragggg/extracted-labels-llama/extracted_labels.json", "r") as f:
    #    extracted = json.load(f)

    #extracted_df = pd.DataFrame.from_dict(extracted, orient="index")
    #extracted_df.index.name = "StudyInstanceUID"
    #extracted_df = extracted_df.reset_index()
    #extracted_df["sample_weight"] = 0.7
    #extracted_df = extracted_df[~extracted_df["StudyInstanceUID"].isin(hard_labeled_df["StudyInstanceUID"])]
    graded_df = pd.read_csv("/kaggle/input/datasets/chiragggg/report-labels-graded/report_labels_graded.csv")
    for col in CFG.TARGETS:
        graded_df[col] = graded_df[col].astype(float)
    extracted_df = graded_df[~graded_df["StudyInstanceUID"].isin(hard_labeled_df["StudyInstanceUID"])]
    labeled_df = pd.concat([hard_labeled_df, extracted_df], ignore_index=True)

    print(f"Hard-labeled: {len(hard_labeled_df)}, Extracted: {len(extracted_df)}, Total: {len(labeled_df)}")
    print(f"Labeled studies : {len(labeled_df)}")
    print(f"Label prevalence:\n{labeled_df[CFG.TARGETS].mean().round(3).to_string()}\n")

    import hashlib

    def hash_report(text):
        return hashlib.md5(str(text).encode()).hexdigest()

    hard_labeled_df["report_hash"] = hard_labeled_df["Report"].apply(hash_report)
    unique_reports = hard_labeled_df.drop_duplicates(subset="report_hash")[["report_hash", "ACL"]]

    skf = StratifiedKFold(n_splits=CFG.FOLDS, shuffle=True, random_state=CFG.SEED)
    hash_to_fold = {}
    for fold, (_, val_idx) in enumerate(skf.split(unique_reports, unique_reports["ACL"])):
        for h in unique_reports.iloc[val_idx]["report_hash"]:
            hash_to_fold[h] = fold

    hard_labeled_df["fold"] = hard_labeled_df["report_hash"].map(hash_to_fold)
    n_dupes = hard_labeled_df["Report"].duplicated().sum()
    print(f"Duplicate reports found in hard-labeled set: {n_dupes}")

    device = torch.device(CFG.DEVICE)
    criterion = nn.BCEWithLogitsLoss(reduction='none')

    for fold in range(CFG.FOLDS):
        print(f"\n{'─'*45}")
        print(f"  Fold {fold + 1}/{CFG.FOLDS}")
        print(f"{'─'*45}")

        train_fold = pd.concat([hard_labeled_df[hard_labeled_df["fold"] != fold], extracted_df], ignore_index=True)
        val_fold = hard_labeled_df[hard_labeled_df["fold"] == fold]

        train_loader = DataLoader(
            RSNADataset(train_fold, series_df, "train", get_transforms(True)),
            batch_size=CFG.BATCH_SIZE, shuffle=True,
            num_workers=CFG.NUM_WORKERS, pin_memory=True,
        )
        val_loader = DataLoader(
            RSNADataset(val_fold, series_df, "train", get_transforms(False)),
            batch_size=CFG.BATCH_SIZE, shuffle=False,
            num_workers=CFG.NUM_WORKERS, pin_memory=True,
        )

        model = RSNAModel().to(device)

        optimizer = torch.optim.AdamW(
            model.parameters(), lr=CFG.LR, weight_decay=CFG.WEIGHT_DECAY
        )
        # Differential learning rates -- real alternative if RSNAModelV2 is
        # ever swapped back in, protecting the pretrained encoder from being
        # pushed as hard as the freshly-initialized head/pool:
        # optimizer = torch.optim.AdamW([
        #     {"params": model.encoder.parameters(), "lr": CFG.LR * 0.1},
        #     {"params": model.pool.parameters(), "lr": CFG.LR},
        #     {"params": model.head.parameters(), "lr": CFG.LR},
        # ], weight_decay=CFG.WEIGHT_DECAY)

        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=CFG.EPOCHS
        )
        scaler = torch.amp.GradScaler()

        best_auc = 0.0
        for epoch in range(CFG.EPOCHS):
            tr_loss = train_one_epoch(model, train_loader, optimizer, criterion, scaler, device)
            val_loss, val_auc = validate(model, val_loader, criterion, device)
            scheduler.step()

            flag = "  ←" if val_auc > best_auc else ""
            print(
                f"  Epoch {epoch+1:02d}  "
                f"train_loss={tr_loss:.4f}  "
                f"val_loss={val_loss:.4f}  "
                f"val_auc={val_auc:.4f}{flag}"
            )

            if val_auc > best_auc:
                best_auc = val_auc
                torch.save(
                    model.state_dict(),
                    CFG.OUTPUT_DIR / f"fold{fold}_best.pth",
                )

        print(f"\n  Best AUC fold {fold+1}: {best_auc:.4f}")
        del model, optimizer, scheduler, scaler
        torch.cuda.empty_cache()


if __name__ == "__main__":
    main()



Hard-labeled: 58, Extracted: 4349, Total: 4407
Labeled studies : 4407
Label prevalence:
ACL                 0.234
MCL                 0.154
Medial Meniscus     0.539
Lateral Meniscus    0.252
Medial OA           0.426
Lateral OA          0.214
PF OA               0.370
Effusion            0.602
Synovitis           0.260
Baker's             0.328
Contusion           0.103
Fracture            0.061

Duplicate reports found in hard-labeled set: 0

─────────────────────────────────────────────
  Fold 1/2
─────────────────────────────────────────────


model.safetensors:   0%|          | 0.00/49.3M [00:00<?, ?B/s]

Prediction std across batch: tensor([0.0042, 0.0067, 0.0083, 0.0057, 0.0029, 0.0028, 0.0062, 0.0079, 0.0033,
        0.0068, 0.0045, 0.0069], device='cuda:0')
Min: tensor([0.4914, 0.5000, 0.4942, 0.4919, 0.5112, 0.4852, 0.4850, 0.4795, 0.5197,
        0.4851, 0.5229, 0.5030], device='cuda:0'), Max: tensor([0.5005, 0.5149, 0.5136, 0.5053, 0.5171, 0.4916, 0.4979, 0.4981, 0.5272,
        0.4994, 0.5338, 0.5176], device='cuda:0')
Encoder gradient norm: 0.2220
Head+pool gradient norm: 0.1018
valid_mask sum: 48.0 of 48
raw_loss sum before mask: 27.098133087158203
raw_loss sum after mask+weight: 22.41289520263672
  Epoch 01  train_loss=0.4277  val_loss=0.6671  val_auc=0.6328  ←

  Best AUC fold 1: 0.6328

─────────────────────────────────────────────
  Fold 2/2
─────────────────────────────────────────────
Prediction std across batch: tensor([0.0051, 0.0033, 0.0022, 0.0060, 0.0045, 0.0081, 0.0095, 0.0048, 0.0073,
        0.0079, 0.0062, 0.0069], device='cuda:0')
Min: tensor([0.5035, 0.4860, 0